In [35]:
import json
import cv2
import numpy as np
from pathlib import Path
from pycocotools import mask as maskUtils
import shutil
from tqdm import tqdm

In [36]:
def validate_coordinates(points):
    """Checks if coordinates in [0, 1] range"""
    return all(0.0 <= x <= 1.0 and 0.0 <= y <= 1.0 for x, y in points)

In [37]:
# Converting dataset from yolo format to coco
def convert_coco_to_yolo(coco_json_path, images_dir, output_dir, class_names):
    with open(coco_json_path) as f:
        data = json.load(f)
    
    output_dir = Path(output_dir)
    (output_dir/'images').mkdir(parents=True, exist_ok=True)
    (output_dir/'labels').mkdir(parents=True, exist_ok=True)
    
    cat_id_map = {cat['id']: class_names.index(cat['name']) for cat in data['categories']}
    
    for img_info in tqdm(data['images'], desc=f"Processing {output_dir.name}"):
        src_path = Path(images_dir)/img_info['file_name']
        dst_path = output_dir/'images'/src_path.name
        
        if not src_path.exists():
            print(f"Missing image: {src_path}")
            continue
            
        shutil.copy(src_path, dst_path)
        
        anns = [a for a in data['annotations'] if a['image_id'] == img_info['id']]
        label_path = output_dir/'labels'/(src_path.stem + '.txt')
        
        with open(label_path, 'w') as f:
            for ann in anns:
                try:
                    # Декодирование маски
                    if isinstance(ann['segmentation'], dict):
                        rle = maskUtils.frPyObjects(
                            ann['segmentation'], 
                            img_info['height'], 
                            img_info['width']
                        )
                        mask = maskUtils.decode(rle)
                    else:
                        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
                        for seg in ann['segmentation']:
                            points = np.array(seg).reshape(-1, 2).round().astype(int)
                            cv2.fillPoly(mask, [points], 1)
                    
                    # Извлечение контуров
                    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    
                    if not contours:
                        continue
                        
                    for contour in contours:
                        epsilon = 0.002 * cv2.arcLength(contour, True)
                        approx = cv2.approxPolyDP(contour, epsilon, True).squeeze()
                        
                        if approx.ndim != 2 or approx.shape[0] < 3:
                            continue
                            
                        # Нормализация
                        approx_norm = approx.astype(np.float32)
                        approx_norm[:, 0] = np.round(approx[:, 0] / img_info['width'], 6)
                        approx_norm[:, 1] = np.round(approx[:, 1] / img_info['height'], 6)
                        
                        # Валидация
                        if not validate_coordinates(approx_norm):
                            continue
                            
                        # Запись
                        points_str = ' '.join([f"{x:.6f} {y:.6f}" for x, y in approx_norm])
                        f.write(f"{cat_id_map[ann['category_id']]} {points_str}\n")
                        
                except Exception as e:
                    print(f"Error processing {src_path.name}: {str(e)}")
                    continue

In [38]:
# Конвертация данных
for split in ['train', 'val', 'test']:
    convert_coco_to_yolo(
        coco_json_path=f"/home/dtsarev/master_of_cv/sem3/DL_project/data/splits_final_deblurred/{split}/labels.json",
        images_dir=f"/home/dtsarev/master_of_cv/sem3/DL_project/data/splits_final_deblurred/{split}/data",
        output_dir=f"./zero_waste_yolo/{split}",
        class_names=['cardboard', 'soft_plastic', 'rigid_plastic', 'metal']
    )

Processing test: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 929/929 [00:11<00:00, 81.20it/s]


In [41]:
# Initialize model
model = YOLO('yolov8n-seg.pt')  # Large segmentation model

In [ ]:
results = model.train(
    data='zerowaste_yolo.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    lr0=1e-3,
    optimizer='Adam',
    augment=True,  # Включаем стандартные аугментации
    overlap_mask=True,
    mask_ratio=2,
    dropout=0.2,
    weight_decay=0.05,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15,
    translate=0.1,
    scale=0.5,
    shear=0.1,
    perspective=0.001,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
)

New https://pypi.org/project/ultralytics/8.3.73 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.50 🚀 Python-3.11.4 torch-2.3.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 11980MiB)
engine/trainer: task=segment, mode=train, model=yolov8n-seg.pt, data=zerowaste_yolo.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=None, name=train5, exist_ok=False, pretrained=True, optimizer=Adam, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=2, dropout=0.2, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=True, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=Fals

train: Scanning /home/dtsarev/master_of_cv/sem3/DL_project/zerowaste/notebooks/zero_waste_yolo/train/labels... 3002 images, 55 backgrounds, 0 corrupt: 100%|██████████| 3002/3002 [00:04<00:00, 714.14it/s]


train: New cache created: /home/dtsarev/master_of_cv/sem3/DL_project/zerowaste/notebooks/zero_waste_yolo/train/labels.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01), CLAHE(p=0.01, clip_limit=(1, 4.0), tile_grid_size=(8, 8))


val: Scanning /home/dtsarev/master_of_cv/sem3/DL_project/zerowaste/notebooks/zero_waste_yolo/val/labels... 572 images, 1 backgrounds, 0 corrupt: 100%|██████████| 572/572 [00:01<00:00, 546.83it/s]

val: New cache created: /home/dtsarev/master_of_cv/sem3/DL_project/zerowaste/notebooks/zero_waste_yolo/val/labels.cache


...

     27/100      3.06G      1.359      2.442      1.808      1.407        127        640: 100%|██████████| 188/188 [00:17<00:00, 10.66it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 18/18 [00:02<00:00,  7.88it/s]


                   all        572       3708      0.355      0.258      0.196       0.11      0.404      0.243      0.198      0.102

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


     28/100      3.19G       1.34      2.426      1.795      1.391        138        640: 100%|██████████| 188/188 [00:17<00:00, 10.71it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 18/18 [00:02<00:00,  7.42it/s]


                   all        572       3708      0.192      0.302      0.165      0.102      0.186      0.286      0.155     0.0796

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


     29/100      3.12G      1.352      2.417      1.791      1.396        166        640:  65%|██████▍   | 122/188 [00:11<00:06, 10.18it/s]